In [18]:
RUN_TARGET = "colab"  # "colab" | "local"


## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1
2. Runtime > Change runtime type > T4 GPU or A100
3. Run the pip-install cell — this installs the notebook dependencies used by the MTL workflow
4. Run the Drive-mount cell — approve access so checkpoints and results can be synced automatically
5. Run the runtime setup cell to download data, the shared utility module, and a fresh ESM-2 snapshot
6. Run the remaining cells normally
7. Outputs are copied to `Google Drive > My Drive > XAllergen2.0 > models/` and `results/`

### Running locally on macOS (M-series)
1. Set `RUN_TARGET = "local"` in Cell 1
2. The Colab setup cells are skipped automatically
3. MPS is used when available, otherwise CPU
4. Outputs are saved to the local `models/` and `results/` directories


In [19]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    # Colab (Python 3.12, torch 2.6+) ships with numpy 2.x, which torch requires.
    # Do NOT pin or downgrade numpy here, or torch / transformers imports can fail
    # with a binary ABI mismatch (for example: "numpy.dtype size changed").
    _required = {
        "captum": "0.8.0",
        "transformers": "4.48.1",
    }
    _optional = ["statsmodels", "huggingface_hub"]

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    for name in _optional:
        try:
            _md.version(name)
        except _md.PackageNotFoundError:
            _missing_or_mismatched.append(name)

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run(
            [
                _sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                *_missing_or_mismatched,
            ],
            check=True,
        )
        print("Colab environment updated. Restart the runtime once, then rerun from the top.")
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Colab environment already compatible. No reinstall needed.


In [20]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    print(f"Google Drive mounted: {DRIVE_ROOT}")
    print(f"Models will sync to: {DRIVE_MODELS}")
    print(f"Results will sync to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted: /content/drive/MyDrive/XAllergen2.0
Models will sync to: /content/drive/MyDrive/XAllergen2.0/models
Results will sync to: /content/drive/MyDrive/XAllergen2.0/results


In [21]:
if RUN_TARGET == "colab":
    import os
    import shutil as _shutil
    import urllib.request as _urlreq
    from pathlib import Path

    from huggingface_hub import snapshot_download

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _DATA_DIR = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    _RESULTS_DIR = RUNTIME_ROOT / "results"
    _FRESH_ESM2_DIR = RUNTIME_ROOT / "hf_models" / "facebook_esm2_t6_8M_UR50D"
    for _d in [_DATA_DIR, _MODEL_DIR, _RESULTS_DIR, _FRESH_ESM2_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    _RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

    _urlreq.urlretrieve(
        f"{_RAW}/baseline_notebook_utils.py",
        RUNTIME_ROOT / "baseline_notebook_utils.py",
    )
    print("Downloaded: baseline_notebook_utils.py")

    for _fname in [
        "positives_splitA.csv",
        "positives_splitB.csv",
        "negatives_splitA.csv",
        "negatives_splitB.csv",
    ]:
        _urlreq.urlretrieve(f"{_RAW}/data/{_fname}", _DATA_DIR / _fname)
        print(f"Downloaded: {_fname}")

    _fresh_path = snapshot_download(
        repo_id="facebook/esm2_t6_8M_UR50D",
        local_dir=_FRESH_ESM2_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ["XALLERGEN_HF_MODEL_DIR"] = str(_fresh_path)
    print(f"Downloaded fresh ESM-2 snapshot: {_fresh_path}")

    _baseline_checkpoint_on_drive = DRIVE_MODELS / "baseline_frozen_esm2.pt"
    if _baseline_checkpoint_on_drive.exists():
        _shutil.copy2(_baseline_checkpoint_on_drive, _MODEL_DIR / _baseline_checkpoint_on_drive.name)
        print(f"Copied baseline checkpoint from Drive: {_baseline_checkpoint_on_drive}")
    else:
        print("No baseline_frozen_esm2.pt found on Drive. Upload or copy it before training the MTL model.")

    _baseline_summary_on_drive = DRIVE_RESULTS / "probing_summary.csv"
    if _baseline_summary_on_drive.exists():
        _shutil.copy2(_baseline_summary_on_drive, _RESULTS_DIR / "probing_summary.csv")
        print(f"Copied baseline probing summary from Drive: {_baseline_summary_on_drive}")
    else:
        print("No probing_summary.csv found on Drive. Baseline-vs-MTL comparison will be skipped.")
else:
    print("Local run: skipping GitHub download and model snapshot setup.")


Downloaded: baseline_notebook_utils.py
Downloaded: positives_splitA.csv
Downloaded: positives_splitB.csv
Downloaded: negatives_splitA.csv
Downloaded: negatives_splitB.csv


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tf_model.h5:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

Downloaded fresh ESM-2 snapshot: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Copied baseline checkpoint from Drive: /content/drive/MyDrive/XAllergen2.0/models/baseline_frozen_esm2.pt
Copied baseline probing summary from Drive: /content/drive/MyDrive/XAllergen2.0/results/probing_summary.csv


# 05 - Multi-Task Epitope Supervision for XAllergen2.0

This notebook fine-tunes the frozen-head portion of `FrozenESMAllergenMTLClassifier` on the curated XAllergen2.0 splits while keeping the ESM-2 backbone frozen.

Training setup:
- Protein-level allergenicity supervision on mixed curated positives and negatives from notebook 01
- Residue-level epitope supervision only on IEDB-derived positive proteins
- Initialization from the saved notebook 03 checkpoint: `models/baseline_frozen_esm2.pt`
- Held-out residue probing on `positives_splitB.csv`

Model:
- Frozen ESM-2 backbone: `esm2_t6_8M_UR50D`
- Reused baseline weights for backbone, `attention_pool`, and protein classifier head
- Fresh epitope head initialized at train time and optimized jointly with the protein head


In [22]:
import os
import sys
from pathlib import Path

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))


In [23]:
import copy
import gc
import json
import math
from pathlib import Path

from baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    MAX_SEQ_LEN,
    RANDOM_STATE,
    THRESHOLD,
    FrozenESMAllergenMTLClassifier,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    compute_residue_probabilities,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    initialize_mtl_from_baseline_checkpoint,
    load_baseline_checkpoint,
    load_mtl_checkpoint,
    mean_metric_dicts,
    parse_epitope_label,
    print_runtime_context,
    seed_everything,
)

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")
else:
    configure_matplotlib_cache(Path.cwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from statsmodels.nonparametric.smoothers_lowess import lowess
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [ ]:
# Hyperparameters
CLASSIFICATION_BATCH_SIZE      = 24
EPOCHS                         = 30
PATIENCE                       = 15
LEARNING_RATE                  = 1e-3
WEIGHT_DECAY                   = 1e-4
LAMBDA_CLS                     = 1.0
LAMBDA_EPI                     = 0.5
EPITOPE_HIDDEN_DIM             = 128
VAL_FRACTION                   = 0.1
USE_PROTEIN_POS_WEIGHT         = False
PROTEIN_IMBALANCE_TOLERANCE    = 0.1
N_RANDOM_DRAWS                 = 100

seed_everything(RANDOM_STATE)

if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: no GPU detected - IG attribution will be slow.")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR    = PROJECT_ROOT / "data"
MODEL_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_TRAIN_CSV      = DATA_DIR / "positives_splitA.csv"
POSITIVE_TEST_CSV       = DATA_DIR / "positives_splitB.csv"
NEGATIVE_TRAIN_CSV      = DATA_DIR / "negatives_splitA.csv"
NEGATIVE_TEST_CSV       = DATA_DIR / "negatives_splitB.csv"
BASELINE_CHECKPOINT_PATH = MODEL_DIR / "baseline_frozen_esm2.pt"
CHECKPOINT_PATH          = MODEL_DIR / "mtl_frozen_esm2_epitope.pt"
METRICS_PATH             = RESULTS_DIR / "mtl_baseline_metrics.json"
PROBE_ROWS_PATH           = RESULTS_DIR / "mtl_probing_rows.csv"
BASELINE_PROBE_ROWS_PATH  = RESULTS_DIR / "baseline_probing_rows.csv"
COMBINED_PROBE_ROWS_PATH  = RESULTS_DIR / "mtl_vs_baseline_probing_rows.csv"
PROBE_SUMMARY_PATH        = RESULTS_DIR / "mtl_probing_summary.csv"
COMPARE_SUMMARY_PATH      = RESULTS_DIR / "mtl_vs_baseline_summary.csv"
COMBINED_VIOLINS_PNG      = RESULTS_DIR / "mtl_vs_baseline_probing_violins.png"
COMBINED_AUROC_DENSITY_PNG = RESULTS_DIR / "mtl_vs_baseline_probing_auroc_vs_density.png"
COMBINED_AUPRC_DENSITY_PNG = RESULTS_DIR / "mtl_vs_baseline_probing_auprc_vs_density.png"
BASELINE_SUMMARY_CSV     = RESULTS_DIR / "probing_summary.csv"

tokenizer = build_tokenizer(HF_MODEL_NAME)


Device: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB


In [25]:
from pathlib import Path

import numpy as np
import pandas as pd

def annotate_epitopes(frame: pd.DataFrame) -> pd.DataFrame:
    annotated = frame.copy()
    annotated["epitope_label"] = [
        parse_epitope_label(seq, start, end)
        for seq, start, end in zip(
            annotated["sequence"],
            annotated["epitope_start"],
            annotated["epitope_end"],
        )
    ]
    annotated["seq_len"] = annotated["sequence"].str.len().astype(int)
    annotated["n_epitope_residues"] = annotated["epitope_label"].map(lambda arr: int(arr.sum()))
    annotated["epitope_density"] = annotated["n_epitope_residues"] / annotated["seq_len"]
    return annotated


def filter_max_len(frame: pd.DataFrame, sequence_col: str = "sequence") -> pd.DataFrame:
    keep = frame[sequence_col].astype(str).str.len() <= MAX_SEQ_LEN
    return frame.loc[keep].reset_index(drop=True)


def audit_frame(csv_path: Path, frame_name: str, sequence_col: str = "sequence") -> tuple[pd.DataFrame, dict]:
    raw = pd.read_csv(csv_path)
    filtered = filter_max_len(raw, sequence_col=sequence_col)
    audit = {
        "frame_name": frame_name,
        "csv_path": str(csv_path),
        "raw_rows": len(raw),
        "kept_rows": len(filtered),
        "dropped_rows": len(raw) - len(filtered),
    }
    return filtered, audit


def print_audit_block(audit_rows: list[dict]) -> None:
    print("Data audit:")
    for row in audit_rows:
        print(
            f"  {row['frame_name']}: raw_rows={row['raw_rows']}, "
            f"kept_rows={row['kept_rows']}, dropped_over_max_len={row['dropped_rows']}"
        )


def require_columns(frame: pd.DataFrame, required: list[str], frame_name: str) -> None:
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns in {frame_name}: {missing}. Available columns: {list(frame.columns)}")


def get_sequence_id_column(frame: pd.DataFrame, preferred: list[str], frame_name: str) -> str:
    for column in preferred:
        if column in frame.columns:
            return column
    raise ValueError(
        f"Could not find a sequence identifier column for {frame_name}. Tried {preferred}. Available columns: {list(frame.columns)}"
    )


def prepare_positive_frame(csv_path: Path, split_name: str, frame_name: str) -> tuple[pd.DataFrame, dict]:
    frame, audit = audit_frame(csv_path, frame_name)
    require_columns(frame, ["sequence", "epitope_start", "epitope_end"], csv_path.name)
    sequence_id_col = get_sequence_id_column(frame, ["accession", "sequence_id"], csv_path.name)
    frame = annotate_epitopes(frame)
    frame = frame.copy()
    frame["sequence_id"] = frame[sequence_id_col].astype(str)
    frame["protein_label"] = 1.0
    frame["has_epitope_supervision"] = 1
    frame["split_name"] = split_name
    frame["data_source"] = "positive"
    return frame, audit


def prepare_negative_frame(csv_path: Path, split_name: str, frame_name: str) -> tuple[pd.DataFrame, dict]:
    frame, audit = audit_frame(csv_path, frame_name)
    require_columns(frame, ["sequence"], csv_path.name)
    sequence_id_col = get_sequence_id_column(frame, ["entry", "sequence_id", "accession"], csv_path.name)
    frame = frame.copy()
    frame["sequence_id"] = frame[sequence_id_col].astype(str)
    frame["seq_len"] = frame["sequence"].str.len().astype(int)
    frame["epitope_label"] = frame["seq_len"].map(lambda seq_len: np.zeros(seq_len, dtype=np.float32))
    frame["n_epitope_residues"] = 0
    frame["epitope_density"] = 0.0
    frame["protein_label"] = 0.0
    frame["has_epitope_supervision"] = 0
    frame["split_name"] = split_name
    frame["data_source"] = "negative"
    return frame, audit


def build_mixed_frame(positive_frame: pd.DataFrame, negative_frame: pd.DataFrame) -> pd.DataFrame:
    columns = [
        "sequence_id",
        "sequence",
        "protein_label",
        "epitope_label",
        "seq_len",
        "has_epitope_supervision",
        "n_epitope_residues",
        "epitope_density",
        "split_name",
        "data_source",
    ]
    mixed = pd.concat(
        [positive_frame[columns], negative_frame[columns]],
        ignore_index=True,
    )
    return mixed.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)


positive_train_full_df, positive_train_audit = prepare_positive_frame(POSITIVE_TRAIN_CSV, "splitA", "positive_train_full")
positive_test_df, positive_test_audit = prepare_positive_frame(POSITIVE_TEST_CSV, "splitB", "positive_test")
negative_train_full_df, negative_train_audit = prepare_negative_frame(NEGATIVE_TRAIN_CSV, "splitA", "negative_train_full")
negative_test_df, negative_test_audit = prepare_negative_frame(NEGATIVE_TEST_CSV, "splitB", "negative_test")

print_audit_block([
    positive_train_audit,
    positive_test_audit,
    negative_train_audit,
    negative_test_audit,
])
print(
    "Post-filter split inputs:",
    f"positive_train_full={len(positive_train_full_df)}",
    f"positive_test={len(positive_test_df)}",
    f"negative_train_full={len(negative_train_full_df)}",
    f"negative_test={len(negative_test_df)}",
)

positive_train_df, positive_val_df = train_test_split(
    positive_train_full_df,
    test_size=VAL_FRACTION,
    random_state=RANDOM_STATE,
)
negative_train_df, negative_val_df = train_test_split(
    negative_train_full_df,
    test_size=VAL_FRACTION,
    random_state=RANDOM_STATE,
)

train_mixed_df = build_mixed_frame(positive_train_df, negative_train_df)
val_mixed_df = build_mixed_frame(positive_val_df, negative_val_df)
test_mixed_df = build_mixed_frame(positive_test_df, negative_test_df)
epitope_probe_df = positive_test_df.copy().reset_index(drop=True)

print("Mixed train/val/test:", len(train_mixed_df), len(val_mixed_df), len(test_mixed_df))
print("Positive train/val/test:", len(positive_train_df), len(positive_val_df), len(positive_test_df))
print("Negative train/val/test:", len(negative_train_df), len(negative_val_df), len(negative_test_df))
print("Positive train density mean:", round(float(positive_train_df["epitope_density"].mean()), 4))
print("Positive test density mean:", round(float(positive_test_df["epitope_density"].mean()), 4))


Mixed train/val/test: 534 61 133
Positive train/val/test: 273 31 60
Negative train/val/test: 261 30 73
Positive train density mean: 0.1942
Positive test density mean: 0.2571


In [26]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

class MixedAllergenEpitopeDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        seq_len = int(row["seq_len"])
        residue_labels = np.asarray(row["epitope_label"], dtype=np.float32)
        if residue_labels.shape[0] != seq_len:
            raise ValueError(
                f"Residue label length mismatch for {row['sequence_id']}: labels={residue_labels.shape[0]}, seq_len={seq_len}"
            )
        return {
            "sequence_id": str(row["sequence_id"]),
            "sequence": row["sequence"],
            "protein_label": float(row["protein_label"]),
            "residue_labels": residue_labels,
            "seq_len": seq_len,
            "has_epitope_supervision": int(row["has_epitope_supervision"]),
            "data_source": row["data_source"],
        }


def collate_mixed_batch(batch: list[dict]) -> dict:
    sequences = [item["sequence"] for item in batch]
    encodings = tokenizer(
        sequences,
        add_special_tokens=False,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    max_len = encodings["input_ids"].shape[1]
    residue_labels = torch.zeros(len(batch), max_len, dtype=torch.float32)
    residue_loss_mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for idx, item in enumerate(batch):
        seq_len = min(item["seq_len"], max_len)
        residue_labels[idx, :seq_len] = torch.tensor(item["residue_labels"][:seq_len], dtype=torch.float32)
        if item["has_epitope_supervision"]:
            residue_loss_mask[idx, :seq_len] = True

    return {
        "sequence_id": [item["sequence_id"] for item in batch],
        "sequence": sequences,
        "protein_label": torch.tensor([item["protein_label"] for item in batch], dtype=torch.float32),
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "residue_labels": residue_labels,
        "residue_loss_mask": residue_loss_mask,
        "has_epitope_supervision": torch.tensor(
            [item["has_epitope_supervision"] for item in batch],
            dtype=torch.float32,
        ),
        "data_source": [item["data_source"] for item in batch],
    }


def move_mixed_batch_to_device(batch: dict, device: str) -> dict:
    moved = dict(batch)
    for key in [
        "protein_label",
        "input_ids",
        "attention_mask",
        "residue_labels",
        "residue_loss_mask",
        "has_epitope_supervision",
    ]:
        moved[key] = batch[key].to(device)
    return moved


train_loader = DataLoader(
    MixedAllergenEpitopeDataset(train_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)
val_loader = DataLoader(
    MixedAllergenEpitopeDataset(val_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)
test_loader = DataLoader(
    MixedAllergenEpitopeDataset(test_mixed_df),
    batch_size=CLASSIFICATION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_mixed_batch,
)


In [27]:
train_positive_proteins = int(positive_train_df["protein_label"].sum())
train_negative_proteins = int((positive_train_df.shape[0] + negative_train_df.shape[0]) - train_positive_proteins)
protein_pos_weight_value = len(negative_train_df) / max(len(positive_train_df), 1)
use_protein_pos_weight = USE_PROTEIN_POS_WEIGHT and abs(protein_pos_weight_value - 1.0) > PROTEIN_IMBALANCE_TOLERANCE
protein_pos_weight = (
    torch.tensor(protein_pos_weight_value, dtype=torch.float32, device=DEVICE)
    if use_protein_pos_weight
    else None
)

total_epitope_residues = float(positive_train_df["n_epitope_residues"].sum())
total_non_epitope_residues = float((positive_train_df["seq_len"] - positive_train_df["n_epitope_residues"]).sum())
pos_weight_epi_value = total_non_epitope_residues / max(total_epitope_residues, 1.0)
residue_pos_weight = torch.tensor(pos_weight_epi_value, dtype=torch.float32, device=DEVICE)

print(f"Training protein positives: {len(positive_train_df)}")
print(f"Training protein negatives: {len(negative_train_df)}")
print(f"Protein pos_weight candidate: {protein_pos_weight_value:.3f}")
print(f"Using protein pos_weight: {use_protein_pos_weight}")
print(f"Training epitope residues (positive set only): {total_epitope_residues:.0f}")
print(f"Training non-epitope residues (positive set only): {total_non_epitope_residues:.0f}")
print(
    "pos_weight_epi = total_non_epitope_residues / total_epitope_residues = "
    f"{total_non_epitope_residues:.0f} / {total_epitope_residues:.0f} = {pos_weight_epi_value:.3f}"
)

if not BASELINE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Baseline checkpoint not found: {BASELINE_CHECKPOINT_PATH}. "
        "Run notebook 03 first or copy baseline_frozen_esm2.pt into models/."
    )

model, baseline_checkpoint = initialize_mtl_from_baseline_checkpoint(
    BASELINE_CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
)

assert not any(param.requires_grad for param in model.backbone.parameters())
trainable_params = [param for param in model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
trainable_parameter_count = int(sum(param.numel() for param in trainable_params))

print(f"Trainable parameter tensors: {len(trainable_params)}")
print(f"Total trainable parameter count: {trainable_parameter_count}")
print(f"Backbone hidden size: {model.backbone.config.hidden_size}")
print(f"Lambda cls: {LAMBDA_CLS}")
print(f"Lambda epi: {LAMBDA_EPI}")


Some weights of EsmModel were not initialized from the model checkpoint at /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training protein positives: 273
Training protein negatives: 261
Protein pos_weight candidate: 0.956
Using protein pos_weight: False
Training epitope residues (positive set only): 14488
Training non-epitope residues (positive set only): 71700
pos_weight_epi = total_non_epitope_residues / total_epitope_residues = 71700 / 14488 = 4.949
Loaded baseline checkpoint: /content/XAllergen2.0/models/baseline_frozen_esm2.pt
Loaded shared keys:
  attention_pool.score.bias
  attention_pool.score.weight
  backbone.contact_head.regression.bias
  backbone.contact_head.regression.weight
  backbone.embeddings.word_embeddings.weight
  backbone.encoder.emb_layer_norm_after.bias
  backbone.encoder.emb_layer_norm_after.weight
  backbone.encoder.layer.0.LayerNorm.bias
  backbone.encoder.layer.0.LayerNorm.weight
  backbone.encoder.layer.0.attention.LayerNorm.bias
  backbone.encoder.layer.0.attention.LayerNorm.weight
  backbone.encoder.layer.0.attention.output.dense.bias
  backbone.encoder.layer.0.attention.out

In [28]:
def compute_protein_loss(
    logits: torch.Tensor,
    protein_labels: torch.Tensor,
    pos_weight: torch.Tensor | None = None,
) -> torch.Tensor:
    kwargs = {"reduction": "mean"}
    if pos_weight is not None:
        kwargs["pos_weight"] = pos_weight
    return F.binary_cross_entropy_with_logits(logits, protein_labels, **kwargs)


def compute_masked_residue_loss(
    residue_logits: torch.Tensor,
    residue_labels: torch.Tensor,
    residue_loss_mask: torch.Tensor,
    pos_weight: torch.Tensor,
) -> tuple[torch.Tensor, int]:
    valid_mask = residue_loss_mask.bool()
    valid_count = int(valid_mask.sum().item())
    if valid_count == 0:
        return residue_logits.sum() * 0.0, 0

    valid_logits = residue_logits[valid_mask]
    valid_labels = residue_labels[valid_mask]
    loss = F.binary_cross_entropy_with_logits(
        valid_logits,
        valid_labels,
        reduction="mean",
        pos_weight=pos_weight,
    )
    return loss, valid_count


def _tensor_min_max(tensor: torch.Tensor) -> tuple[float, float]:
    detached = tensor.detach()
    return float(detached.min().cpu().item()), float(detached.max().cpu().item())


def _raise_if_non_finite_losses(
    stage: str,
    batch: dict,
    outputs: dict[str, torch.Tensor],
    cls_loss: torch.Tensor,
    epi_loss: torch.Tensor,
    total_loss: torch.Tensor,
) -> None:
    non_finite_losses = [
        name
        for name, value in {"cls_loss": cls_loss, "epi_loss": epi_loss, "total_loss": total_loss}.items()
        if not torch.isfinite(value).all()
    ]
    if not non_finite_losses:
        return

    batch_size = int(batch["protein_label"].shape[0])
    valid_positions = int(batch["residue_loss_mask"].bool().sum().item())
    logits_min, logits_max = _tensor_min_max(outputs["logits"])
    residue_logits_min, residue_logits_max = _tensor_min_max(outputs["residue_logits"])
    sequence_ids = list(batch.get("sequence_id", []))
    if sequence_ids:
        print(f"[{stage}] Non-finite loss batch sequence_ids: {sequence_ids}")

    raise ValueError(
        f"[{stage}] Non-finite loss detected in {', '.join(non_finite_losses)} | "
        f"cls_loss={float(cls_loss.detach().cpu().item())} | "
        f"epi_loss={float(epi_loss.detach().cpu().item())} | "
        f"total_loss={float(total_loss.detach().cpu().item())} | "
        f"batch_size={batch_size} | "
        f"valid_residue_positions={valid_positions} | "
        f"logits_min={logits_min} | logits_max={logits_max} | "
        f"residue_logits_min={residue_logits_min} | residue_logits_max={residue_logits_max} | "
        f"sequence_ids={sequence_ids}"
    )


def train_one_epoch_mtl(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    protein_pos_weight: torch.Tensor | None,
    residue_pos_weight: torch.Tensor,
    lambda_cls: float,
    lambda_epi: float,
    trainable_params: list[torch.nn.Parameter],
) -> dict:
    model.train()
    total_cls_numerator = 0.0
    total_cls_examples = 0
    total_epi_numerator = 0.0
    total_epi_positions = 0
    total_total_loss = 0.0
    total_steps = 0

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch["input_ids"], batch["attention_mask"])
        cls_loss = compute_protein_loss(outputs["logits"], batch["protein_label"], protein_pos_weight)
        epi_loss, epi_positions = compute_masked_residue_loss(
            outputs["residue_logits"],
            batch["residue_labels"],
            batch["residue_loss_mask"],
            residue_pos_weight,
        )
        total_loss = lambda_cls * cls_loss + lambda_epi * epi_loss
        _raise_if_non_finite_losses("train", batch, outputs, cls_loss, epi_loss, total_loss)

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()

        batch_size = int(batch["protein_label"].shape[0])
        total_cls_numerator += float(cls_loss.item()) * batch_size
        total_cls_examples += batch_size
        if epi_positions > 0:
            total_epi_numerator += float(epi_loss.item()) * epi_positions
            total_epi_positions += epi_positions
        total_total_loss += float(total_loss.item())
        total_steps += 1

    train_cls_loss = total_cls_numerator / max(total_cls_examples, 1)
    train_epi_loss = total_epi_numerator / max(total_epi_positions, 1)
    train_total_loss = total_total_loss / max(total_steps, 1)
    return {
        "train_total_loss": train_total_loss,
        "train_cls_loss": train_cls_loss,
        "train_epi_loss": train_epi_loss,
        "train_weighted_cls": lambda_cls * train_cls_loss,
        "train_weighted_epi": lambda_epi * train_epi_loss,
    }


@torch.no_grad()
def evaluate_mtl(
    model: nn.Module,
    loader: DataLoader,
    device: str,
    protein_pos_weight: torch.Tensor | None,
    residue_pos_weight: torch.Tensor,
    lambda_cls: float,
    lambda_epi: float,
) -> dict:
    model.eval()
    total_cls_numerator = 0.0
    total_cls_examples = 0
    total_epi_numerator = 0.0
    total_epi_positions = 0
    total_total_loss = 0.0
    total_steps = 0

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        cls_loss = compute_protein_loss(outputs["logits"], batch["protein_label"], protein_pos_weight)
        epi_loss, epi_positions = compute_masked_residue_loss(
            outputs["residue_logits"],
            batch["residue_labels"],
            batch["residue_loss_mask"],
            residue_pos_weight,
        )
        total_loss = lambda_cls * cls_loss + lambda_epi * epi_loss
        _raise_if_non_finite_losses("eval", batch, outputs, cls_loss, epi_loss, total_loss)

        batch_size = int(batch["protein_label"].shape[0])
        total_cls_numerator += float(cls_loss.item()) * batch_size
        total_cls_examples += batch_size
        if epi_positions > 0:
            total_epi_numerator += float(epi_loss.item()) * epi_positions
            total_epi_positions += epi_positions
        total_total_loss += float(total_loss.item())
        total_steps += 1

    cls_loss_value = total_cls_numerator / max(total_cls_examples, 1)
    epi_loss_value = total_epi_numerator / max(total_epi_positions, 1)
    total_loss_value = total_total_loss / max(total_steps, 1)
    return {
        "total_loss": total_loss_value,
        "cls_loss": cls_loss_value,
        "epi_loss": epi_loss_value,
        "weighted_cls": lambda_cls * cls_loss_value,
        "weighted_epi": lambda_epi * epi_loss_value,
    }


@torch.no_grad()
def predict_mtl(
    model: nn.Module,
    loader: DataLoader,
    device: str,
) -> tuple[pd.DataFrame, dict]:
    model.eval()
    protein_rows = []
    residue_predictions = []
    flat_residue_labels = []
    flat_residue_scores = []

    for batch in loader:
        batch = move_mixed_batch_to_device(batch, device)
        outputs = model(batch["input_ids"], batch["attention_mask"])
        protein_probs = torch.sigmoid(outputs["logits"]).detach().cpu()
        residue_probs = torch.sigmoid(outputs["residue_logits"]).detach().cpu()
        attention_mask = batch["attention_mask"].detach().cpu()
        residue_labels = batch["residue_labels"].detach().cpu()
        has_supervision = batch["has_epitope_supervision"].detach().cpu()
        protein_labels = batch["protein_label"].detach().cpu()

        for idx, sequence_id in enumerate(batch["sequence_id"]):
            prob = float(protein_probs[idx].item())
            protein_rows.append(
                {
                    "sequence_id": sequence_id,
                    "sequence": batch["sequence"][idx],
                    "label": int(protein_labels[idx].item()),
                    "pred_prob": prob,
                    "pred_label": int(prob >= THRESHOLD),
                    "logit": float(outputs["logits"][idx].detach().cpu().item()),
                }
            )

            if int(has_supervision[idx].item()) == 1:
                seq_len = int(attention_mask[idx].sum().item())
                labels = residue_labels[idx, :seq_len].numpy().astype(np.float32)
                scores = residue_probs[idx, :seq_len].numpy().astype(np.float32)
                residue_predictions.append(
                    {
                        "sequence_id": sequence_id,
                        "residue_labels": labels,
                        "residue_scores": scores,
                    }
                )
                flat_residue_labels.append(labels)
                flat_residue_scores.append(scores)

    payload = {
        "residue_predictions": residue_predictions,
        "residue_labels_flat": np.concatenate(flat_residue_labels) if flat_residue_labels else np.array([], dtype=np.float32),
        "residue_scores_flat": np.concatenate(flat_residue_scores) if flat_residue_scores else np.array([], dtype=np.float32),
    }
    return pd.DataFrame(protein_rows), payload


def compute_classification_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label"].to_numpy()
    y_prob = pred_df["pred_prob"].to_numpy()
    y_pred = pred_df["pred_label"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "auroc": float(roc_auc_score(y_true, y_prob)),
        "auprc": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else math.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }


def compute_flattened_residue_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    n_valid = int(labels.shape[0])
    n_positive = int(labels.sum())

    if n_valid == 0 or n_positive == 0 or n_positive == n_valid:
        return {
            "n_valid_residues": n_valid,
            "n_positive_residues": n_positive,
            "auroc": math.nan,
            "auprc": math.nan,
            "precision_at_k": math.nan,
        }

    k = max(n_positive, 1)
    top_k = np.argsort(scores)[-k:]
    return {
        "n_valid_residues": n_valid,
        "n_positive_residues": n_positive,
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "precision_at_k": float(labels[top_k].mean()),
    }


In [29]:
history = []
best_val_total = float("inf")
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False

for epoch in tqdm(range(1, EPOCHS + 1), desc="Training", unit="epoch"):
    train_stats = train_one_epoch_mtl(
        model,
        train_loader,
        optimizer,
        DEVICE,
        protein_pos_weight=protein_pos_weight,
        residue_pos_weight=residue_pos_weight,
        lambda_cls=LAMBDA_CLS,
        lambda_epi=LAMBDA_EPI,
        trainable_params=trainable_params,
    )
    val_stats = evaluate_mtl(
        model,
        val_loader,
        DEVICE,
        protein_pos_weight=protein_pos_weight,
        residue_pos_weight=residue_pos_weight,
        lambda_cls=LAMBDA_CLS,
        lambda_epi=LAMBDA_EPI,
    )

    row = {
        "epoch": epoch,
        "train_total_loss": float(train_stats["train_total_loss"]),
        "train_cls_loss": float(train_stats["train_cls_loss"]),
        "train_epi_loss": float(train_stats["train_epi_loss"]),
        "train_weighted_cls": float(train_stats["train_weighted_cls"]),
        "train_weighted_epi": float(train_stats["train_weighted_epi"]),
        "val_total_loss": float(val_stats["total_loss"]),
        "val_cls_loss": float(val_stats["cls_loss"]),
        "val_epi_loss": float(val_stats["epi_loss"]),
        "val_weighted_cls": float(val_stats["weighted_cls"]),
        "val_weighted_epi": float(val_stats["weighted_epi"]),
    }
    history.append(row)

    if val_stats["total_loss"] < best_val_total:
        best_val_total = float(val_stats["total_loss"])
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "esm_model_name": ESM_MODEL_NAME,
                "baseline_checkpoint_path": str(BASELINE_CHECKPOINT_PATH),
                "architecture_hyperparameters": {
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                    "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
                },
                "training_history": history,
                "best_epoch": best_epoch,
                "lambda_cls": LAMBDA_CLS,
                "lambda_epi": LAMBDA_EPI,
                "protein_pos_weight": None if protein_pos_weight is None else float(protein_pos_weight.item()),
                "residue_pos_weight": float(residue_pos_weight.item()),
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:>3}/{EPOCHS} | "
        f"train_total={train_stats['train_total_loss']:.5f} | "
        f"train_cls={train_stats['train_cls_loss']:.5f} | "
        f"train_epi={train_stats['train_epi_loss']:.5f} | "
        f"train_lambda_cls={train_stats['train_weighted_cls']:.5f} | "
        f"train_lambda_epi={train_stats['train_weighted_epi']:.5f} | "
        f"val_total={val_stats['total_loss']:.5f} | "
        f"val_cls={val_stats['cls_loss']:.5f} | "
        f"val_epi={val_stats['epi_loss']:.5f} | "
        f"val_lambda_cls={val_stats['weighted_cls']:.5f} | "
        f"val_lambda_epi={val_stats['weighted_epi']:.5f} | "
        f"best={best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        early_stopped = True
        print(f"Early stopping triggered at epoch {epoch}.")
        break

print(f"Best validation objective: {best_val_total:.5f} at epoch {best_epoch}")
print(f"Early stopping triggered: {early_stopped}")
print(f"Checkpoint saved to: {CHECKPOINT_PATH}")


Training:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_total=0.99957 | train_cls=0.42084 | train_epi=1.14221 | train_lambda_cls=0.42084 | train_lambda_epi=0.57111 | val_total=1.04860 | val_cls=0.40856 | val_epi=1.29963 | val_lambda_cls=0.40856 | val_lambda_epi=0.64982 | best=1
Epoch   2/30 | train_total=0.91443 | train_cls=0.36347 | train_epi=1.09847 | train_lambda_cls=0.36347 | train_lambda_epi=0.54923 | val_total=1.02067 | val_cls=0.34894 | val_epi=1.35739 | val_lambda_cls=0.34894 | val_lambda_epi=0.67869 | best=2
Epoch   3/30 | train_total=0.86902 | train_cls=0.32883 | train_epi=1.08080 | train_lambda_cls=0.32883 | train_lambda_epi=0.54040 | val_total=0.99660 | val_cls=0.32737 | val_epi=1.35753 | val_lambda_cls=0.32737 | val_lambda_epi=0.67877 | best=3
Epoch   4/30 | train_total=0.81636 | train_cls=0.28598 | train_epi=1.06082 | train_lambda_cls=0.28598 | train_lambda_epi=0.53041 | val_total=0.94957 | val_cls=0.32700 | val_epi=1.26610 | val_lambda_cls=0.32700 | val_lambda_epi=0.63305 | best=4
Epoch   5/30 | train_tot

In [30]:
model, checkpoint = load_mtl_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    epitope_hidden_dim=EPITOPE_HIDDEN_DIM,
)

history_df = pd.DataFrame(checkpoint["training_history"])
best_epoch = int(checkpoint.get("best_epoch", int(history_df.loc[history_df["val_total_loss"].idxmin(), "epoch"])))
early_stopped_flag = bool(globals().get("early_stopped", False))

val_stats = evaluate_mtl(
    model,
    val_loader,
    DEVICE,
    protein_pos_weight=protein_pos_weight,
    residue_pos_weight=residue_pos_weight,
    lambda_cls=LAMBDA_CLS,
    lambda_epi=LAMBDA_EPI,
)
test_stats = evaluate_mtl(
    model,
    test_loader,
    DEVICE,
    protein_pos_weight=protein_pos_weight,
    residue_pos_weight=residue_pos_weight,
    lambda_cls=LAMBDA_CLS,
    lambda_epi=LAMBDA_EPI,
)

val_predictions_df, val_residue_payload = predict_mtl(model, val_loader, DEVICE)
test_predictions_df, test_residue_payload = predict_mtl(model, test_loader, DEVICE)

val_classification_metrics = compute_classification_metrics(val_predictions_df)
val_residue_metrics = compute_flattened_residue_metrics(
    val_residue_payload["residue_labels_flat"],
    val_residue_payload["residue_scores_flat"],
)
test_metrics = compute_classification_metrics(test_predictions_df)
test_metrics["test_total_loss"] = float(test_stats["total_loss"])
test_metrics["test_cls_loss"] = float(test_stats["cls_loss"])
test_metrics["test_epi_loss"] = float(test_stats["epi_loss"])
test_metrics["test_weighted_cls"] = float(test_stats["weighted_cls"])
test_metrics["test_weighted_epi"] = float(test_stats["weighted_epi"])
test_metrics["best_epoch"] = best_epoch
test_metrics["n_test_sequences"] = int(len(test_predictions_df))

test_residue_metrics = compute_flattened_residue_metrics(
    test_residue_payload["residue_labels_flat"],
    test_residue_payload["residue_scores_flat"],
)

metrics_payload = {
    "esm_model_name": ESM_MODEL_NAME,
    "baseline_checkpoint_path": str(BASELINE_CHECKPOINT_PATH),
    "architecture_hyperparameters": {
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
        "epitope_hidden_dim": EPITOPE_HIDDEN_DIM,
    },
    "training": {
        "batch_size": CLASSIFICATION_BATCH_SIZE,
        "epochs_requested": EPOCHS,
        "early_stopping_patience": PATIENCE,
        "optimizer": "AdamW",
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "lambda_cls": LAMBDA_CLS,
        "lambda_epi": LAMBDA_EPI,
        "use_protein_pos_weight": use_protein_pos_weight,
        "protein_pos_weight": None if protein_pos_weight is None else float(protein_pos_weight.item()),
        "residue_pos_weight": float(residue_pos_weight.item()),
        "residue_pos_weight_formula": "total_non_epitope_residues / total_epitope_residues",
        "total_epitope_residues_train": float(total_epitope_residues),
        "total_non_epitope_residues_train": float(total_non_epitope_residues),
        "best_epoch": best_epoch,
        "early_stopped": early_stopped_flag,
    },
    "validation_losses": {
        "total_loss": float(val_stats["total_loss"]),
        "cls_loss": float(val_stats["cls_loss"]),
        "epi_loss": float(val_stats["epi_loss"]),
        "weighted_cls": float(val_stats["weighted_cls"]),
        "weighted_epi": float(val_stats["weighted_epi"]),
    },
    "validation_classification_metrics": val_classification_metrics,
    "validation_residue_metrics": val_residue_metrics,
    "test_metrics": test_metrics,
    "test_residue_metrics": test_residue_metrics,
}

with METRICS_PATH.open("w") as handle:
    json.dump(metrics_payload, handle, indent=2)

print("Validation classification metrics:")
print(json.dumps(val_classification_metrics, indent=2))
print("Validation residue metrics:")
print(json.dumps(val_residue_metrics, indent=2))
print("Test classification metrics:")
print(json.dumps(test_metrics, indent=2))
print("Test residue metrics:")
print(json.dumps(test_residue_metrics, indent=2))
print(f"Saved metrics to: {METRICS_PATH}")


Some weights of EsmModel were not initialized from the model checkpoint at /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Validation classification metrics:
{
  "threshold": 0.5,
  "accuracy": 0.8688524590163934,
  "auroc": 0.9430107526881719,
  "auprc": 0.9452979683137452,
  "f1": 0.875,
  "precision": 0.8484848484848485,
  "recall": 0.9032258064516129,
  "specificity": 0.8333333333333334,
  "mcc": 0.738946695932336,
  "confusion_matrix": {
    "tn": 25,
    "fp": 5,
    "fn": 3,
    "tp": 28
  }
}
Validation residue metrics:
{
  "n_valid_residues": 8534,
  "n_positive_residues": 2052,
  "auroc": 0.6440905028349612,
  "auprc": 0.3291329767164074,
  "precision_at_k": 0.3328460156917572
}
Test classification metrics:
{
  "threshold": 0.5,
  "accuracy": 0.8721804511278195,
  "auroc": 0.9436073059360729,
  "auprc": 0.9400144382227498,
  "f1": 0.8521739130434782,
  "precision": 0.8909090909090909,
  "recall": 0.8166666666666667,
  "specificity": 0.9178082191780822,
  "mcc": 0.7421391791638636,
  "confusion_matrix": {
    "tn": 67,
    "fp": 6,
    "fn": 11,
    "tp": 49
  },
  "test_total_loss": 0.95428490638

In [31]:
def compute_probe_metrics(labels: np.ndarray, scores: np.ndarray) -> dict:
    labels = np.asarray(labels, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)
    seq_len = len(labels)
    positives = int(labels.sum())

    if seq_len == 0 or positives == 0 or positives == seq_len:
        return {"auroc": np.nan, "auprc": np.nan, "precision_at_k": np.nan}

    auroc = roc_auc_score(labels, scores)
    auprc = average_precision_score(labels, scores)

    k = max(positives, 1)
    top_k = np.argsort(scores)[-k:]
    precision_at_k = float(labels[top_k].mean())

    return {
        "auroc": float(auroc),
        "auprc": float(auprc),
        "precision_at_k": precision_at_k,
    }


IG_PROBE_DEVICE = "cpu"
IG_INTERNAL_BATCH_SIZE = 1
rng = np.random.default_rng(RANDOM_STATE)
probe_rows = []
baseline_probe_rows = []
ig_model = copy.deepcopy(model).to(IG_PROBE_DEVICE)
ig_model.eval()
baseline_model, _ = load_baseline_checkpoint(
    BASELINE_CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)
baseline_ig_model = copy.deepcopy(baseline_model).to(IG_PROBE_DEVICE)
baseline_ig_model.eval()

print(f"Integrated Gradients device: {IG_PROBE_DEVICE}")
print(f"IG_STEPS: {IG_STEPS}")
print(f"IG internal_batch_size: {IG_INTERNAL_BATCH_SIZE}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

for _, row in tqdm(epitope_probe_df.iterrows(), total=len(epitope_probe_df), desc="Probing splitB"):
    sequence = row["sequence"]
    epitope_labels = row["epitope_label"]
    base = {
        "accession": row["accession"],
        "seq_len": int(row["seq_len"]),
        "epitope_density": float(row["epitope_density"]),
        "n_epitope_residues": int(row["n_epitope_residues"]),
    }

    residue_scores = compute_residue_probabilities(model, tokenizer, sequence, DEVICE)
    probe_rows.append(
        {**base, "model_family": "MTL (05)", "method": "residue_head", **compute_probe_metrics(epitope_labels, residue_scores)}
    )

    attention_scores = compute_attention_weights(model, tokenizer, sequence, DEVICE)
    probe_rows.append(
        {**base, "model_family": "MTL (05)", "method": "attention_weights", **compute_probe_metrics(epitope_labels, attention_scores)}
    )

    baseline_attention_scores = compute_attention_weights(baseline_model, tokenizer, sequence, DEVICE)
    baseline_probe_rows.append(
        {**base, "model_family": "Baseline (04)", "method": "attention_weights", **compute_probe_metrics(epitope_labels, baseline_attention_scores)}
    )

    ig_scores = compute_integrated_gradients(
        ig_model,
        tokenizer,
        sequence,
        IG_PROBE_DEVICE,
        steps=IG_STEPS,
        normalize=False,
        internal_batch_size=IG_INTERNAL_BATCH_SIZE,
    )
    probe_rows.append(
        {**base, "model_family": "MTL (05)", "method": "integrated_gradients", **compute_probe_metrics(epitope_labels, ig_scores)}
    )

    baseline_ig_scores = compute_integrated_gradients(
        baseline_ig_model,
        tokenizer,
        sequence,
        IG_PROBE_DEVICE,
        steps=IG_STEPS,
        normalize=False,
        internal_batch_size=IG_INTERNAL_BATCH_SIZE,
    )
    baseline_probe_rows.append(
        {**base, "model_family": "Baseline (04)", "method": "integrated_gradients", **compute_probe_metrics(epitope_labels, baseline_ig_scores)}
    )

    random_metrics = [
        compute_probe_metrics(epitope_labels, rng.uniform(0.0, 1.0, size=len(epitope_labels)))
        for _ in range(N_RANDOM_DRAWS)
    ]
    random_summary = mean_metric_dicts(random_metrics)
    probe_rows.append({**base, "model_family": "MTL (05)", "method": "random_mean", **random_summary})
    baseline_probe_rows.append({**base, "model_family": "Baseline (04)", "method": "random_mean", **random_summary})

    del residue_scores, attention_scores, baseline_attention_scores, ig_scores, baseline_ig_scores, random_metrics, random_summary
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

del ig_model
del baseline_ig_model
del baseline_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

probe_df = pd.DataFrame(probe_rows)
baseline_probe_df = pd.DataFrame(baseline_probe_rows)
combined_probe_df = pd.concat([baseline_probe_df, probe_df], ignore_index=True)

probe_df.to_csv(PROBE_ROWS_PATH, index=False)
baseline_probe_df.to_csv(BASELINE_PROBE_ROWS_PATH, index=False)
combined_probe_df.to_csv(COMBINED_PROBE_ROWS_PATH, index=False)
print(f"Saved MTL row-wise probe metrics to: {PROBE_ROWS_PATH}")
print(f"Saved baseline row-wise probe metrics to: {BASELINE_PROBE_ROWS_PATH}")
print(f"Saved combined row-wise probe metrics to: {COMBINED_PROBE_ROWS_PATH}")
probe_df.head()


Integrated Gradients device: cpu
IG_STEPS: 50
IG internal_batch_size: 1


Probing splitB:   0%|          | 0/60 [00:00<?, ?it/s]

Saved row-wise probe metrics to: /content/XAllergen2.0/results/mtl_probing_rows.csv


,accession,seq_len,epitope_density,n_epitope_residues,method,auroc,auprc,precision_at_k
0,A0A834K244,251,0.549801,138,residue_head,0.531871,0.553572,0.608696
1,A0A834K244,251,0.549801,138,attention_weights,0.538540,0.563661,0.550725
2,A0A834K244,251,0.549801,138,integrated_gradients,0.378222,0.457163,0.456522
3,A0A834K244,251,0.549801,138,random_mean,0.496637,0.555965,0.547971
4,I1KMV0,165,0.090909,15,residue_head,0.865333,0.581740,0.600000


In [32]:
def summarize_probe_methods(frame: pd.DataFrame, methods: list[str]) -> pd.DataFrame:
    summary_rows = []
    for method in methods:
        method_df = frame[frame["method"] == method]
        summary_rows.append(
            {
                "method": method,
                "auroc_mean": round(float(method_df["auroc"].dropna().mean()), 4),
                "auroc_sd": round(float(method_df["auroc"].dropna().std()), 4),
                "auprc_mean": round(float(method_df["auprc"].dropna().mean()), 4),
                "auprc_sd": round(float(method_df["auprc"].dropna().std()), 4),
                "precision_at_k_mean": round(float(method_df["precision_at_k"].dropna().mean()), 4),
                "precision_at_k_sd": round(float(method_df["precision_at_k"].dropna().std()), 4),
                "n_proteins": int(len(method_df)),
            }
        )
    return pd.DataFrame(summary_rows)

summary_df = summarize_probe_methods(
    probe_df,
    ["residue_head", "attention_weights", "integrated_gradients", "random_mean"],
)
baseline_summary_df = summarize_probe_methods(
    baseline_probe_df,
    ["attention_weights", "integrated_gradients", "random_mean"],
)
summary_df.to_csv(PROBE_SUMMARY_PATH, index=False)
print(summary_df.to_string(index=False))
print(f"Saved summary to: {PROBE_SUMMARY_PATH}")

comparable_methods = ["attention_weights", "integrated_gradients", "random_mean"]
comparison_df = baseline_summary_df.merge(
    summary_df[summary_df["method"].isin(comparable_methods)],
    on="method",
    suffixes=("_baseline", "_mtl"),
)
for metric in ["auroc_mean", "auprc_mean", "precision_at_k_mean"]:
    comparison_df[f"delta_{metric}"] = (
        comparison_df[f"{metric}_mtl"] - comparison_df[f"{metric}_baseline"]
    ).round(4)
comparison_df.to_csv(COMPARE_SUMMARY_PATH, index=False)
print("\nBaseline vs MTL comparison")
print(comparison_df.to_string(index=False))
print(f"Saved comparison to: {COMPARE_SUMMARY_PATH}")


              method  auroc_mean  auroc_sd  auprc_mean  auprc_sd  precision_at_k_mean  precision_at_k_sd  n_proteins
        residue_head      0.6415    0.1592      0.4089    0.2562               0.3754             0.2482          60
   attention_weights      0.4594    0.1445      0.2583    0.1751               0.2289             0.1848          60
integrated_gradients      0.4837    0.1656      0.2729    0.1864               0.2470             0.2011          60
         random_mean      0.5007    0.0056      0.2766    0.1739               0.2579             0.1731          60
Saved summary to: /content/XAllergen2.0/results/mtl_probing_summary.csv

Baseline vs MTL comparison
              method  auroc_mean_baseline  auroc_sd_baseline  auprc_mean_baseline  auprc_sd_baseline  precision_at_k_mean_baseline  precision_at_k_sd_baseline  n_proteins_baseline  auroc_mean_mtl  auroc_sd_mtl  auprc_mean_mtl  auprc_sd_mtl  precision_at_k_mean_mtl  precision_at_k_sd_mtl  n_proteins_mtl  delta_auro

In [33]:
PALETTE = {
    "attention_weights": "#4C72B0",
    "integrated_gradients": "#DD8452",
    "random_mean": "#55A868",
    "residue_head": "#C44E52",
}
METHOD_XLABELS = {
    "attention_weights": "Attention\nWeights",
    "integrated_gradients": "Integrated\nGradients",
    "random_mean": "Random\nMean",
    "residue_head": "Residue\nHead (MTL)",
}
FAMILY_ORDER = ["Baseline (04)", "MTL (05)"]
FAMILY_LINESTYLE = {"Baseline (04)": "--", "MTL (05)": "-"}

violin_order = ["attention_weights", "integrated_gradients", "random_mean", "residue_head"]
violin_df = combined_probe_df[combined_probe_df["method"].isin(violin_order)].copy()
metrics_config = [
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
    ("precision_at_k", "Precision@k"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (col, label) in zip(axes, metrics_config):
    plot_data = violin_df.dropna(subset=[col]).copy()

    sns.violinplot(
        data=plot_data,
        x="method",
        y=col,
        hue="model_family",
        order=violin_order,
        hue_order=FAMILY_ORDER,
        inner=None,
        cut=0,
        dodge=True,
        ax=ax,
    )
    sns.stripplot(
        data=plot_data,
        x="method",
        y=col,
        hue="model_family",
        order=violin_order,
        hue_order=FAMILY_ORDER,
        dodge=True,
        alpha=0.35,
        size=3.5,
        jitter=True,
        ax=ax,
    )

    for family in FAMILY_ORDER:
        family_random = plot_data[(plot_data["method"] == "random_mean") & (plot_data["model_family"] == family)][col]
        if not family_random.empty:
            ax.axhline(
                float(family_random.mean()),
                color="gray",
                linestyle=FAMILY_LINESTYLE[family],
                linewidth=1.2,
                alpha=0.9,
            )

    overall_mean = plot_data[col].mean()
    overall_sd = plot_data[col].std()
    ax.set_title(f"{label}\nmean±SD: {overall_mean:.3f} ± {overall_sd:.3f}", fontsize=11)
    ax.set_xlabel("Method")
    ax.set_ylabel(label)
    ax.set_xticklabels([METHOD_XLABELS[m] for m in violin_order], fontsize=9)

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    if ax is axes[0]:
        ax.legend(unique.values(), unique.keys(), title="Model family", fontsize=8)
    else:
        ax.legend().remove()

plt.suptitle("Residue Attribution Faithfulness vs. IEDB Epitopes: Baseline vs MTL", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(COMBINED_VIOLINS_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved plot to: {COMBINED_VIOLINS_PNG}")


Saved plot to: /content/XAllergen2.0/results/mtl_probing_means.png


In [35]:
FAMILY_MARKER = {"Baseline (04)": "o", "MTL (05)": "^"}
scatter_methods = ["attention_weights", "integrated_gradients", "random_mean", "residue_head"]
scatter_df = combined_probe_df[combined_probe_df["method"].isin(scatter_methods)].copy()

for metric_col, metric_label, out_path in [
    ("auroc", "AUROC", COMBINED_AUROC_DENSITY_PNG),
    ("auprc", "AUPRC", COMBINED_AUPRC_DENSITY_PNG),
]:
    fig, ax = plt.subplots(figsize=(9, 6))

    for method in scatter_methods:
        for family in FAMILY_ORDER:
            mdf = scatter_df[
                (scatter_df["method"] == method) & (scatter_df["model_family"] == family)
            ].dropna(subset=[metric_col, "epitope_density"])
            if mdf.empty:
                continue

            color = PALETTE[method]
            ax.scatter(
                mdf["epitope_density"],
                mdf[metric_col],
                color=color,
                alpha=0.38,
                s=30,
                marker=FAMILY_MARKER[family],
                label=f"{METHOD_XLABELS[method].replace(chr(10), ' ')} — {family}",
            )
            if len(mdf) >= 5:
                smoothed = lowess(
                    mdf[metric_col].values,
                    mdf["epitope_density"].values,
                    frac=0.5,
                    return_sorted=True,
                )
                ax.plot(
                    smoothed[:, 0],
                    smoothed[:, 1],
                    color=color,
                    linewidth=2.0,
                    linestyle=FAMILY_LINESTYLE[family],
                )

    ax.set_xlabel("Epitope Density (fraction of residues)", fontsize=12)
    ax.set_ylabel(metric_label, fontsize=12)
    ax.set_title(f"{metric_label} vs. Epitope Density", fontsize=13)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {out_path}")


Saved comparison plot to: /content/XAllergen2.0/results/mtl_vs_baseline_plot.png


In [36]:
if RUN_TARGET == "colab":
    import shutil as _shutil

    DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

    for _src, _dst_dir in [
        (CHECKPOINT_PATH, DRIVE_MODELS),
        (METRICS_PATH, DRIVE_RESULTS),
        (PROBE_ROWS_PATH, DRIVE_RESULTS),
        (BASELINE_PROBE_ROWS_PATH, DRIVE_RESULTS),
        (COMBINED_PROBE_ROWS_PATH, DRIVE_RESULTS),
        (PROBE_SUMMARY_PATH, DRIVE_RESULTS),
        (COMPARE_SUMMARY_PATH, DRIVE_RESULTS),
        (COMBINED_VIOLINS_PNG, DRIVE_RESULTS),
        (COMBINED_AUROC_DENSITY_PNG, DRIVE_RESULTS),
        (COMBINED_AUPRC_DENSITY_PNG, DRIVE_RESULTS),
    ]:
        if _src.exists():
            _shutil.copy2(_src, _dst_dir / _src.name)
            print(f"Copied to Drive: {_dst_dir / _src.name}")
        else:
            print(f"Skipped missing output: {_src}")
else:
    print("Local run: outputs saved to:")
    for _out_path in [
        CHECKPOINT_PATH,
        METRICS_PATH,
        PROBE_ROWS_PATH,
        BASELINE_PROBE_ROWS_PATH,
        COMBINED_PROBE_ROWS_PATH,
        PROBE_SUMMARY_PATH,
        COMPARE_SUMMARY_PATH,
        COMBINED_VIOLINS_PNG,
        COMBINED_AUROC_DENSITY_PNG,
        COMBINED_AUPRC_DENSITY_PNG,
    ]:
        print(f"  {_out_path}")


Copied to Drive: /content/drive/MyDrive/XAllergen2.0/models/mtl_frozen_esm2_epitope.pt
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_baseline_metrics.json
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_probing_rows.csv
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_probing_summary.csv
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_vs_baseline_summary.csv
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_probing_means.png
Copied to Drive: /content/drive/MyDrive/XAllergen2.0/results/mtl_vs_baseline_plot.png


## Interpretation Guide

What to look for after running the notebook:
- `results/mtl_baseline_metrics.json`: sequence-level and flattened residue-level metrics for the checkpoint initialized from notebook 03.
- `train_cls_loss` vs `train_epi_loss` and their weighted counterparts: confirms whether protein classification or residue supervision is dominating optimization.
- `validation_residue_metrics` and `test_residue_metrics`: flattened residue AUROC/AUPRC over valid positive residues only.
- `results/mtl_probing_summary.csv`: held-out per-protein alignment scores on `positives_splitB.csv` for the residue head, attention weights, and integrated gradients.
- `results/mtl_vs_baseline_summary.csv`: change in attention- and IG-based alignment relative to notebook 03.
- `results/mtl_vs_baseline_plot.png`: side-by-side bar plot comparing notebook 04 baseline/random probing with notebook 05 MTL probing.
- If the residue head improves while protein metrics stay stable, the auxiliary task is adding localized supervision without sacrificing allergen classification.
